### ЗАДАЧА: Пакетная загрузка накладных (try/except + custom exceptions)

Из внешней системы приходят строки с накладными.
Нужно безопасно разобрать их, отделить валидные записи от ошибок и собрать краткий отчёт.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `InvoiceError`
   - `RowFormatError`
   - `QuantityError`
   - `PriceError`
   - `StatusError`.

2. Функцию `parse_invoice(row)`:
   - формат: `invoice_id,item,quantity,price,status`
   - `quantity` должен быть целым числом и `> 0`
   - `price` должен быть числом и `> 0`
   - допустимые статусы: `new`, `approved`, `paid`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `load_invoices(rows)`:
   - вернуть `(invoices, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных накладных,
   - ошибки по типам,
   - сумму только для накладных со статусом `paid`,
   - товар-лидер по суммарному количеству в валидных накладных.

ПОДСКАЗКИ:
- `quantity` и `price` сначала конвертируются, потом валидируются.
- Для ошибок по типам удобно собирать обычный словарь-счётчик.


In [4]:
rows = [
    'INV-100,Keyboard,3,120,paid',
    'INV-101,Mouse,0,40,new',
    'INV-102,Monitor,2,abc,approved',
    'INV-103,Laptop,1,1400,shipped',
    'INV-104,Keyboard,5,110,paid',
    'INV-105,Dock,2,-50,approved',
]


class InvoiceError(Exception):
    pass


class RowFormatError(InvoiceError):
    pass


class QuantityError(InvoiceError):
    pass


class PriceError(InvoiceError):
    pass


class StatusError(InvoiceError):
    pass


def parse_invoice(row):
    # TODO: распарсить строку и провалидировать quantity, price, status
    # TODO: при ошибках конвертации использовать raise ... from ...
    arr = row.split(',')
    if len(arr) != 5:
        raise InvoiceError('Неподходящее количество елементов')
    try:
        q = int(arr[2])
    except ValueError as e:
        raise QuantityError('количество должно быть числом') from e
    if q < 0:
        raise QuantityError("количество должно быть больше 0")
    try:
        p = int(arr[3])
    except ValueError as e:
        raise PriceError('цена должна быть числом') from e
    if p < 0:
        raise PriceError('цена должна быть больше 0')
    
    if arr[4] not in ('new', 'approved', 'paid'):
        raise StatusError('некорректный статус')
    return {'invoice_id': arr[0], 'item':arr[1], 'quantity':q, 'price':p,'status':arr[4]}
    





def load_invoices(rows):
    # TODO: вернуть (invoices, errors)
    invoices = []
    errors = []
    for el in rows:
        try:
            invoices.append(parse_invoice(el))
        except Exception as e:
            errors.append((el, type(e).__name__, e))
    return (invoices, errors)
    


# TODO: вызвать load_invoices(rows)
# TODO: вывести число валидных накладных и число ошибок
# TODO: вывести ошибки по типам
# TODO: посчитать paid_total
# TODO: найти товар-лидер по количеству
res = load_invoices(rows)
count_valid = 0
count_invalid = 0
for el in res[0]:
    count_valid += 1
for el in res[1]:
    count_invalid += 1

type_errors = {}
for el in res[1]:
    type_errors.setdefault(el[1], [])
    type_errors[el[1]].append(el)

paid_total = 0
for el in res[0]:
    paid_total += el['quantity'] * el['price']

m = 0
lider = None

for el in res[0]:
    if el['quantity'] > m:
        m = el['quantity']
        lider = el['item']
    
print(f'Количество валидных записей: {count_valid}')
print(f'Количество невалидных записей: {count_invalid}')
print(type_errors)
print(f'Сумма цен: {paid_total}')
print(f'Товар-лидер по количеству: {lider}')






Количество валидных записей: 3
Количество невалидных записей: 3
{'PriceError': [('INV-102,Monitor,2,abc,approved', 'PriceError', PriceError('цена должна быть числом')), ('INV-105,Dock,2,-50,approved', 'PriceError', PriceError('цена должна быть больше 0'))], 'StatusError': [('INV-103,Laptop,1,1400,shipped', 'StatusError', StatusError('некорректный статус'))]}
Сумма цен: 910
Товар-лидер по количеству: Keyboard
